# This is a sample Jupyter Notebook

Below is an example of a code cell. 
Put your cursor into the cell and press Shift+Enter to execute it and select the next one, or click 'Run Cell' button.

Press Double Shift to search everywhere for classes, files, tool windows, actions, and settings.

To learn more about Jupyter Notebooks in PyCharm, see [help](https://www.jetbrains.com/help/pycharm/ipython-notebook-support.html).
For an overview of PyCharm, go to Help -> Learn IDE features or refer to [our documentation](https://www.jetbrains.com/help/pycharm/getting-started.html).

In [11]:
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_absolute_error

# 1. Load Data and Separate Target (Same as before)
df = pd.read_csv('Food_Expiration_Dates_dataset.csv')
df_encoded = pd.get_dummies(df, columns=['Category'])
X = df_encoded.drop('Spoilage_Days', axis=1)
y = df_encoded['Spoilage_Days']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. GRID SEARCH SETUP (The part that impresses the jury)
# We provide a dictionary of hyperparameters for the system to test
param_grid = {
    'max_depth': [3, 5, 8],            # Depth of the trees
    'learning_rate': [0.01, 0.1, 0.2], # Step size shrinkage
    'n_estimators': [100, 300, 500]    # Number of boosting rounds (trees)
}

print("Initiating Grid Search. Testing 27 different combinations (This may take 1-2 minutes)...")

# Base XGBoost Model
base_model = xgb.XGBRegressor(objective='reg:squarederror', random_state=42)

# Initialize GridSearchCV with 3-fold cross-validation (cv=3)
grid_search = GridSearchCV(estimator=base_model, param_grid=param_grid,
                           scoring='neg_mean_absolute_error', cv=3, verbose=1)

# Start training and finding the best parameters
grid_search.fit(X_train, y_train)

print("\n--- OPTIMIZATION COMPLETED ---")
print("Mathematically Best Parameters Found:", grid_search.best_params_)

# 3. Extract the Best Model and Evaluate
best_xgb_model = grid_search.best_estimator_
predictions = best_xgb_model.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
print(f"\nFinal Mean Absolute Error (MAE): {mae:.2f} days")

# 4. Save the Optimized Model for Deployment
best_xgb_model.save_model('xgboost_spoilage_model.json')
print("Model saved successfully as 'xgboost_spoilage_model.json'")

Initiating Grid Search. Testing 27 different combinations (This may take 1-2 minutes)...
Fitting 3 folds for each of 27 candidates, totalling 81 fits

--- OPTIMIZATION COMPLETED ---
Mathematically Best Parameters Found: {'learning_rate': 0.01, 'max_depth': 5, 'n_estimators': 500}

Final Mean Absolute Error (MAE): 3.43 days
Model saved successfully as 'xgboost_spoilage_model.json'
